# Elliptic Dataset: Temporal Graph Neural Network

Can temporal graph neural networks improve Bitcoin fraud detection by modeling the evolution of the transaction graph over time compared with static graph neural networks and traditional machine learning models?

This notebook evaluates whether a temporal graph neural network can improve illicit transaction detection by treating each Elliptic time step as a separate graph snapshot.

Unlike the static GNN notebook, which trains on one graph with temporal masks, this model processes the graph as a sequence of snapshots. This makes the experiment closer to the real structure of the Elliptic dataset.

Important limitation: the Elliptic graph has no cross-time-step edges, so the temporal GNN is learning changes in graph structure and feature distributions across time rather than explicit transaction paths across time.


### Torch-Geometric-Temporal

Torch-Geometric-Temporal extends PyTorch Geometric with neural network architectures designed for dynamic graphs. These models capture how node representations evolve over time rather than assuming a static graph.

In [1]:
!pip install torch-geometric-temporal -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.5 MB/s eta 0:00:00


Note: Installing the torch-geometric-temporal library in a fresh Google Colab environment required approximately 26 minutes. This setup time is a one-time dependency installation and is not included in the model runtimes reported throughout this notebook.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import networkx as nx

import os
import copy
import math
import random

import torch
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch_geometric_temporal.nn.recurrent import EvolveGCNH, EvolveGCNO

import torch.nn as nn
import torch_geometric.nn as tgnn

import time


In [3]:


from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
results_dir = '/content/drive/MyDrive/Elliptic/results/'
if not os.path.exists(results_dir):
    os.makedirs(results_dir, exist_ok=True)
print(f"Results directory set to: {results_dir}")

Results directory set to: /content/drive/MyDrive/Elliptic/results/


In [5]:
df_classes = pd.read_csv('/content/drive/MyDrive/elliptic_txs_classes.csv')
df_classes['class'] = pd.to_numeric(df_classes['class'], errors = 'coerce')
df_classes['class'] = df_classes['class'].map({1: 'illicit', 2: 'licit'}).fillna('unknown')

df_edgelist = pd.read_csv('/content/drive/MyDrive/elliptic_txs_edgelist.csv')
df_edgelist = df_edgelist.rename(columns={'txId1': 'source', 'txId2': 'target'})


df_features = pd.read_parquet("/content/drive/MyDrive/Elliptic/processed/nodes/clean.parquet")

comparison_03 = pd.read_csv(
    "/content/drive/MyDrive/Elliptic/results/03_graph_feature_engineering_comparison.csv"
)

comparison_04 = pd.read_csv(
    "/content/drive/MyDrive/Elliptic/results/04_static_gnn_comparison.csv"
)

## Temporal Split

This notebook uses the same temporal train, validation, and test split introduced in `03_Graph_Feature_Engineering.ipynb`. Maintaining a consistent temporal split across Notebooks 01–05 ensures that model performance can be compared fairly and that all experiments are reproducible.

In [6]:
train_df = df_features[df_features["time_step"] <= 34].copy()

val_df = df_features[
    (df_features["time_step"] >= 35) &
    (df_features["time_step"] <= 39)
].copy()

test_df = df_features[df_features["time_step"] >= 40].copy()

print(f"Training set shape: {train_df.shape}")
print(f"Validaton set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

print(f"Training set Time Step Range: {train_df['time_step'].min(), train_df['time_step'].max()}")
print(f"Validation set Time Step Range: {val_df['time_step'].min(), val_df['time_step'].max()}")
print(f"Test set Time Step Range: {test_df['time_step'].min(), test_df['time_step'].max()}")

Training set shape: (136265, 172)
Validaton set shape: (20857, 172)
Test set shape: (46647, 172)
Training set Time Step Range: (1, 34)
Validation set Time Step Range: (35, 39)
Test set Time Step Range: (40, 49)


### Basic Setup

The objective is to predict whether each transaction is illicit or licit using temporal graph snapshots constructed from the Elliptic dataset.

In [7]:
label_col = 'label'
time_col = 'time_step'
tx_col = 'txId'

# keep only known labels for evaluation/training
# Filter by string labels 'illicit' and 'licit'
df_known = df_features[df_features[label_col].isin(['illicit', 'licit'])].copy()

# convert labels to binary: ilicit -> 1, licit -> 0
df_known['y'] = df_known[label_col].map({'illicit': 1, 'licit': 0}).astype(int)

features_cols = [
    c for c in df_features.columns
    if c not in [tx_col, time_col, label_col, 'y']
]

print('Num features columns:', len(features_cols))
print("Known rows:", df_known.shape)


Num features columns: 169
Known rows: (46564, 173)


### Load or Build Temporal Graph Snapshots

Temporal GNNs operate on a sequence of graphs rather than a single static graph. Therefore, each Elliptic time step is converted into its own graph snapshot. Because constructing these snapshots is computationally expensive, they are saved to disk and reused in future runs.

In [8]:
snapshots_path = os.path.join(results_dir, 'temporal_snapshots.pt')

snapshots = []

if os.path.exists(snapshots_path):
    try:
        snapshots = torch.load(snapshots_path)
        print(f"Snapshots loaded from {snapshots_path}. Total {len(snapshots)} snapshots.")
    except Exception as e:
        print(f"Error loading snapshots: {e}. Re-generating snapshots.")
        # Fallback to generation if loading fails
        snapshots = [] # Clear snapshots if partial load occurred
else:
    print(f"Snapshots file not found at {snapshots_path}. Generating snapshots.")

# Only generate if snapshots list is empty (i.e., not loaded successfully)
if not snapshots:
    for t in sorted(df_known[time_col].unique()):
        df_t = df_known[df_known[time_col] == t].copy()

        tx_ids_t = df_t[tx_col].values
        tx_to_idx = {tx: i for i, tx in enumerate(tx_ids_t)}

        # keep only edges where both endpoints are inside the same time step
        edges_t = df_edgelist[
            df_edgelist["source"].isin(tx_ids_t) &
            df_edgelist["target"].isin(tx_ids_t)
        ].copy()

        if len(edges_t) > 0:
            edge_index = torch.tensor(
                [
                    edges_t["source"].map(tx_to_idx).values,
                    edges_t['target'].map(tx_to_idx).values
                ],
                dtype=torch.long
            )

            # make undirected for GCN-style messeage passing
            edge_index_rev = edge_index[[1, 0], :]
            edge_index = torch.cat([edge_index, edge_index_rev], dim=1)
        else:
            edge_index = torch.empty((2, 0), dtype=torch.long)

        x = torch.tensor(df_t[features_cols].values, dtype=torch.float)
        y = torch.tensor(df_t['y'].values, dtype=torch.long)

        snapshots.append({
            "time_step": int(t),
            "x": x,
            "edge_index": edge_index,
            "y": y,
            "num_nodes": x.shape[0],
            "num_edges": edge_index.shape[1]
        })

    # Save snapshots after generation if they were just generated
    try:
        torch.save(snapshots, snapshots_path)
        print(f"Snapshots generated and saved to {snapshots_path}")
    except Exception as e:
        print(f"Error saving generated snapshots: {e}")

print("Num Snapshots:", len(snapshots))
if snapshots:
    print(snapshots[0]['time_step'], snapshots[0]['x'].shape, snapshots[0]['edge_index'].shape)

Snapshots loaded from /content/drive/MyDrive/Elliptic/results/temporal_snapshots.pt. Total 49 snapshots.
Num Snapshots: 49
1 torch.Size([2147, 169]) torch.Size([2, 3848])


### Saving and Loading Intermediate Data

To reduce runtime and avoid repeating computationally expensive preprocessing, we save intermediate results, such as the `snapshots` list, to disk. This allows previously generated data to be loaded directly instead of being recreated each time the notebook is executed.

Because `snapshots` contains PyTorch tensors, `torch.save` and `torch.load` provide an efficient and reliable way to serialize and restore the data. A conditional check is included to load the saved file only if it already exists; otherwise, the snapshots are generated and saved for future use.

In [9]:
snapshots_path = os.path.join(results_dir, 'temporal_snapshots.pt') # Use results_dir for consistency

try:
    torch.save(snapshots, snapshots_path)
    print(f"Snapshots saved to {snapshots_path}")
except Exception as e:
    print(f"Error saving snapshots: {e}")


loaded_snapshots = None
if os.path.exists(snapshots_path):
    try:
        loaded_snapshots = torch.load(snapshots_path)
        print(f"Snapshots loaded from {snapshots_path}")

    except Exception as e:
        print(f"Error loading snapshots: {e}")
else:
    print(f"Snapshots file not found at {snapshots_path}. Will generate them if needed.")

Snapshots saved to /content/drive/MyDrive/Elliptic/results/temporal_snapshots.pt
Snapshots loaded from /content/drive/MyDrive/Elliptic/results/temporal_snapshots.pt


## Temporal Split

This will dividing the dataset into training, validation and testing based on time. The snapshots will create a new list.

In [10]:
train_steps = set(range(1,35))
val_steps = set(range(35, 40))
test_steps = set(range(40, 50))

train_snapshots = [s for s in snapshots if s["time_step"] in train_steps]
val_snapshots = [s for s in snapshots if s['time_step'] in val_steps]
test_snapshots = [s for s in snapshots if s['time_step'] in test_steps]

print("Train:", [s["time_step"] for s in train_snapshots])
print('Val:', [s['time_step'] for s in val_snapshots])
print('Test:', [s['time_step'] for s in test_snapshots])

Train: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
Val: [35, 36, 37, 38, 39]
Test: [40, 41, 42, 43, 44, 45, 46, 47, 48, 49]


## EvolveGCN-H

Financial transaction networks naturally evolve over time as new transactions and relationships are formed. Unlike static graph neural networks, EvolveGCN-H models these temporal dynamics by evolving the hidden representations of the graph convolutional network across sequential graph snapshots. This approach allows node embeddings to incorporate both graph structure and temporal information.

To ensure reproducibility, random seeds are fixed for Python, NumPy, and PyTorch. The EvolveGCN-H model is then implemented as a PyTorch neural network consisting of a recurrent graph convolutional layer, two fully connected layers, ReLU activation functions, and dropout regularization. The model configuration specifies the input, hidden, and output dimensions required for binary node classification.

Training is performed using the Adam optimizer with a weighted cross-entropy loss function to address the class imbalance between illicit and licit transactions. During evaluation, the model processes each temporal graph snapshot, performs a forward pass, predicts node labels, and computes accuracy, precision, recall, and F1-score, with particular emphasis on the illicit transaction class.

#### Hidden-State Dimension

The hidden-state dimension was set to 64 to provide a moderate level of model capacity while maintaining computational efficiency. This value was retained as a fixed experimental setting to support reproducibility and to keep the comparison between EvolveGCN-H and EvolveGCN-O focused on architectural differences rather than extensive hyperparameter tuning.

Although the hidden-state dimension can substantially affect the amount of temporal information represented by the model, identifying the globally optimal dimension was outside the scope of this notebook. Therefore, the reported results should be interpreted as a controlled architecture comparison rather than a fully optimized benchmark.

In [11]:
# Seed control

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# EvolveGCN-H Model

class EvolveGCNClassifier(torch.nn.Module):
  def __init__(self, num_of_nodes, in_channels, hidden_channels, num_classes):
    super().__init__()
    self.recurrent = EvolveGCNH(
        num_of_nodes = num_of_nodes, # Re-adding num_of_nodes as it's required
        in_channels = in_channels
    )

    self.linear1 = torch.nn.Linear(in_channels, hidden_channels)
    self.linear2 = torch.nn.Linear(hidden_channels, num_classes)

  def forward(self, x, edge_index):
    h = self.recurrent(x, edge_index)
    h = F.relu(self.linear1(h))
    h = F.dropout(h, p=0.30, training=self.training)
    out = self.linear2(h)
    return out

# Helper function

def reset_recurrent_state_h(model):
    if hasattr(model.recurrent, "H"):
        model.recurrent.H = None
    if hasattr(model.recurrent, "_H"):
        model.recurrent._H = None
    if hasattr(model.recurrent, "_cached_h"):
        model.recurrent._cached_h = None
    if hasattr(model.recurrent, "weight"):
        model.recurrent.weight = None

# Training Setup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

in_channels = train_snapshots[0]["x"].shape[1]
hidden_channels = 64
num_classes = 2
num_of_nodes = 1

model_h = EvolveGCNClassifier(
    num_of_nodes=num_of_nodes,
    in_channels=in_channels,
    hidden_channels=hidden_channels,
    num_classes=num_classes
).to(device)

optimizer_h = torch.optim.Adam(
    model_h.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

# Class weights, and masked version

all_train_y = []

for snap in train_snapshots:
    y = snap["y"]
    mask = y >= 0
    all_train_y.append(y[mask])

all_train_y = torch.cat(all_train_y)

class_counts = torch.bincount(all_train_y)
class_weights = class_counts.sum() / (2 * class_counts.float())
class_weights = class_weights.to(device)

criterion_h = torch.nn.CrossEntropyLoss(weight=class_weights)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

# Evaluation Function

@torch.no_grad()
def evaluate_h(model, snapshot_list):
    model.eval()
    reset_recurrent_state_h(model)

    y_true_all = []
    y_pred_all = []

    for snap in snapshot_list:
        x = snap["x"].to(device)
        edge_index = snap["edge_index"].to(device)
        y = snap["y"].to(device)

        mask = y >= 0

        out = model(x, edge_index)
        pred = out.argmax(dim=1)

        y_true_all.extend(y[mask].cpu().numpy())
        y_pred_all.extend(pred[mask].cpu().numpy())

    acc = accuracy_score(y_true_all, y_pred_all)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_all,
        y_pred_all,
        labels=[1],
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": acc,
        "illicit_precision": precision,
        "illicit_recall": recall,
        "illicit_f1": f1
    }

# Train

best_state_h = None
best_val_f1_h = -1
num_epochs = 40

# start time

start_time_h = time.perf_counter()

# Training loop

for epoch in range(1, num_epochs + 1):
    model_h.train()
    total_loss = 0

    for snap in train_snapshots:
        reset_recurrent_state_h(model_h)   # reset inside snapshot loop

        x = snap["x"].to(device)
        edge_index = snap["edge_index"].to(device)
        y = snap["y"].to(device)

        mask = y >= 0

        optimizer_h.zero_grad()
        out = model_h(x, edge_index)

        loss = criterion_h(out[mask], y[mask])
        loss.backward()
        optimizer_h.step()

        total_loss += loss.item()

    val_metrics = evaluate_h(model_h, val_snapshots)
    val_f1 = val_metrics["illicit_f1"]

    if val_f1 > best_val_f1_h:
        best_val_f1_h = val_f1
        best_state_h = copy.deepcopy(model_h.state_dict())

    print(f"Epoch {epoch:03d} | Loss: {total_loss:.4f} | Val F1: {val_f1:.4f}")

    runtime_minutes_h = (time.perf_counter() - start_time_h) / 60

    print(f"\nEvolveGCN-O training runtime: {runtime_minutes_h:.3f} minutes")

cpu
Class counts: tensor([26432,  3462])
Class weights: tensor([0.5655, 4.3174])
Epoch 001 | Loss: 21.0128 | Val F1: 0.2211

EvolveGCN-O training runtime: 0.017 minutes
Epoch 002 | Loss: 21.4085 | Val F1: 0.2627

EvolveGCN-O training runtime: 0.027 minutes
Epoch 003 | Loss: 16.6914 | Val F1: 0.3493

EvolveGCN-O training runtime: 0.038 minutes
Epoch 004 | Loss: 12.5662 | Val F1: 0.3994

EvolveGCN-O training runtime: 0.048 minutes
Epoch 005 | Loss: 11.4718 | Val F1: 0.3945

EvolveGCN-O training runtime: 0.058 minutes
Epoch 006 | Loss: 10.6727 | Val F1: 0.3884

EvolveGCN-O training runtime: 0.068 minutes
Epoch 007 | Loss: 10.1817 | Val F1: 0.3567

EvolveGCN-O training runtime: 0.079 minutes
Epoch 008 | Loss: 9.4932 | Val F1: 0.3674

EvolveGCN-O training runtime: 0.089 minutes
Epoch 009 | Loss: 9.4206 | Val F1: 0.3828

EvolveGCN-O training runtime: 0.099 minutes
Epoch 010 | Loss: 9.1488 | Val F1: 0.2814

EvolveGCN-O training runtime: 0.112 minutes
Epoch 011 | Loss: 9.1169 | Val F1: 0.3605


### Final validaton and test

In [12]:
model_h.load_state_dict(best_state_h)

reset_recurrent_state_h(model_h)
val_metrics_h = evaluate_h(model_h, val_snapshots)

reset_recurrent_state_h(model_h)
test_metrics_h = evaluate_h(model_h, test_snapshots)

print("Validation (EvolveGCN-H):")
print(val_metrics_h)

print("\nTest (EvolveGCN-H):")
print(test_metrics_h)

Validation (EvolveGCN-H):
{'accuracy': 0.909952606635071, 'illicit_precision': 0.46444780635400906, 'illicit_recall': 0.6868008948545862, 'illicit_f1': 0.5541516245487365}

Test (EvolveGCN-H):
{'accuracy': 0.7896995708154506, 'illicit_precision': 0.071, 'illicit_recall': 0.22327044025157233, 'illicit_f1': 0.10773899848254932}


## Save for EvolveGCN-H

In [13]:
results_h = pd.DataFrame([
    {
        "Feature Set": "Temporal Graph Snapshots",
        "Model": "EvolveGCN-H",
        "Num Features": in_channels,
        "Split": "validation",
        "Accuracy": val_metrics_h["accuracy"],
        "Illicit Precision": val_metrics_h["illicit_precision"],
        "Illicit Recall": val_metrics_h["illicit_recall"],
        "Illicit F1": val_metrics_h["illicit_f1"],
        "Runtime Minutes": runtime_minutes_h
    },
    {
        "Feature Set": "Temporal Graph Snapshots",
        "Model": "EvolveGCN-H",
        "Num Features": in_channels,
        "Split": "test",
        "Accuracy": test_metrics_h["accuracy"],
        "Illicit Precision": test_metrics_h["illicit_precision"],
        "Illicit Recall": test_metrics_h["illicit_recall"],
        "Illicit F1": test_metrics_h["illicit_f1"],
        "Runtime Minutes": runtime_minutes_h
    }
])

results_h

,Feature Set,Model,Num Features,Split,Accuracy,Illicit Precision,Illicit Recall,Illicit F1,Runtime Minutes
0,Temporal Graph Snapshots,EvolveGCN-H,169,validation,0.909953,0.464448,0.686801,0.554152,0.477907
1,Temporal Graph Snapshots,EvolveGCN-H,169,test,0.789700,0.071000,0.223270,0.107739,0.477907


### Results

The EvolveGCN-H model achieved an illicit F1 score of `0.554` on the validation set and `0.108` on the test set. Compared with the previous machine learning and graph neural network models, these results indicate weaker generalization to unseen temporal data.

We initialized the EvolveGCN-H model using a Google Colab GPU and evaluated its performance on the validation and test datasets. The total training and evaluation runtime was `0.478` minutes, which was substantially faster than the previous models evaluated in this project. Although EvolveGCN-H successfully modeled temporal information, its validation and test performance remained below the Random Forest baseline. These results suggest that evolving the hidden node representations may not capture the temporal characteristics of the Elliptic dataset as effectively as expected. Therefore, the next experiment evaluates EvolveGCN-O, which evolves the graph convolutional weights directly rather than the hidden node representations.

## EvolveGCN-O

In [14]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

# Model EvloveGCN-O

class EvolveGCNOClassifier(nn.Module):
  def __init__(self, in_channels, num_classes=2, dropout=0.30):
    super().__init__()
    self.evolvegcn_o = EvolveGCNO(in_channels=in_channels)
    self.dropout = dropout
    self.classifier = nn.Linear(in_channels, num_classes)

  def forward(self, x, edge_index, edge_weight=None):
    h = self.evolvegcn_o(x, edge_index, edge_weight)
    h = F.relu(h)
    h = F.dropout(h, p=self.dropout, training=self.training)
    out = self.classifier(h)
    return out

# recurrent-state reset helper

def reset_recurrent_state_o(model):
  # EvolveGCNO stores its state in _cached_h, and might have others like H, C, weight
  if hasattr(model.evolvegcn_o, "_cached_h"):
    model.evolvegcn_o._cached_h = None
  if hasattr(model.evolvegcn_o, "H"):
    model.evolvegcn_o.H = None
  if hasattr(model.evolvegcn_o, "C"):
    model.evolvegcn_o.C = None
  if hasattr(model.evolvegcn_o, "weight"):
    model.evolvegcn_o.weight = None

# Class weights for train snapshots

all_train_y = []

for snap in train_snapshots:
  y = snap['y']
  mask = y >= 0
  all_train_y.append(y[mask])

all_train_y = torch.cat(all_train_y)

num_licit = (all_train_y == 0).sum().item()
num_illicit = (all_train_y == 1).sum().item()

class_weight = torch.tensor(
    [
        len(all_train_y) / (2 * num_licit),
        len(all_train_y) / (2 * num_illicit)
    ],
    dtype=torch.float
).to(device)

class_weight

# initalize model

in_channels = train_snapshots[0]['x'].shape[1]

model_o = EvolveGCNOClassifier(
    in_channels=in_channels,
    num_classes=2,
    dropout=0.30
).to(device)

optimizer_o = torch.optim.Adam(
    model_o.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

criterion = nn.CrossEntropyLoss(weight=class_weight)

# Evaliation

@torch.no_grad()

def evaluate_o(model, snapshot_list):
  model.eval()
  reset_recurrent_state_o(model)

  y_true_all = []
  y_pred_all = []

  for snap in snapshot_list:
    x = snap['x'].to(device)
    edge_index = snap['edge_index'].to(device)
    y = snap['y'].to(device)

    mask = y >= 0

    out = model(x, edge_index)
    pred = out.argmax(dim=1)

    y_true_all.extend(y[mask].cpu().numpy())
    y_pred_all.extend(pred[mask].cpu().numpy())

  acc = accuracy_score(y_true_all, y_pred_all)

  precision, recall, f1, _ = precision_recall_fscore_support(
      y_true_all,
      y_pred_all,
      labels=[1],
      average = 'binary',
      zero_division=0
  )

  return {
      'accuracy': acc,
      'illicit_precision': precision,
      'illicit_recall': recall,
      'illicit_f1': f1
  }

# Training Loop

num_epochs = 40
best_val_f1_o = -1
best_state_o = None # Re-initialize best_state for EvolveGCNOClassifier

# start time

start_time_o = time.perf_counter()

history = []

for epoch in range(1, num_epochs + 1 ):
  model_o.train()
  # Do not reset here, reset will be done per snapshot

  total_loss = 0

  for snap in train_snapshots:
    reset_recurrent_state_o(model_o) # Reset recurrent state for each snapshot

    x = snap['x'].to(device)
    edge_index = snap['edge_index'].to(device)
    y = snap['y'].to(device)

    mask = y >= 0

    optimizer_o.zero_grad()

    out = model_o(x, edge_index)
    loss = criterion(out[mask], y[mask])

    loss.backward()
    optimizer_o.step()

    total_loss += loss.item()

  # Reset before evaluation as well
  reset_recurrent_state_o(model_o)
  val_metrics = evaluate_o(model_o, val_snapshots)

  history.append({
      'epoch': epoch,
      'loss': total_loss,
      **val_metrics
  })

  if val_metrics['illicit_f1'] > best_val_f1_o:
    best_val_f1_o = val_metrics['illicit_f1']
    best_state_o = copy.deepcopy(model_o.state_dict())

  print(
      f"Epoch {epoch:03d} | "
      f"Loss: {total_loss:.4} | "
      f"Val F1: {val_metrics['illicit_f1']:.4f} | "
      f"Precision: {val_metrics['illicit_precision']:.4f} | "
      f"Recall: {val_metrics['illicit_recall']:.4f}"
  )

  runtime_minutes_o = (time.perf_counter() - start_time_o) / 60

  print(f"\nEvolveGCN-O training runtime: {runtime_minutes_o:.3f} minutes")

Epoch 001 | Loss: 17.78 | Val F1: 0.3444 | Precision: 0.2265 | Recall: 0.7181

EvolveGCN-O training runtime: 0.010 minutes
Epoch 002 | Loss: 16.08 | Val F1: 0.3619 | Precision: 0.2370 | Recall: 0.7651

EvolveGCN-O training runtime: 0.019 minutes
Epoch 003 | Loss: 10.26 | Val F1: 0.3534 | Precision: 0.2276 | Recall: 0.7897

EvolveGCN-O training runtime: 0.029 minutes
Epoch 004 | Loss: 9.497 | Val F1: 0.3569 | Precision: 0.2315 | Recall: 0.7785

EvolveGCN-O training runtime: 0.039 minutes
Epoch 005 | Loss: 9.033 | Val F1: 0.3641 | Precision: 0.2378 | Recall: 0.7763

EvolveGCN-O training runtime: 0.049 minutes
Epoch 006 | Loss: 8.519 | Val F1: 0.3773 | Precision: 0.2496 | Recall: 0.7718

EvolveGCN-O training runtime: 0.058 minutes
Epoch 007 | Loss: 8.029 | Val F1: 0.3958 | Precision: 0.2667 | Recall: 0.7673

EvolveGCN-O training runtime: 0.068 minutes
Epoch 008 | Loss: 7.684 | Val F1: 0.4170 | Precision: 0.2909 | Recall: 0.7360

EvolveGCN-O training runtime: 0.077 minutes
Epoch 009 | Loss

In [15]:
# --- Final validation/test evaluation ---
model_o.load_state_dict(best_state_o)

val_metrics_o = evaluate_o(model_o, val_snapshots)
test_metrics_o = evaluate_o(model_o, test_snapshots)

print("Validation (EvolveGCN-O):")
print(val_metrics_o)

print("\nTest (EvolveGCN-O):")
print(test_metrics_o)


Validation (EvolveGCN-O):
{'accuracy': 0.94896099161502, 'illicit_precision': 0.6902050113895216, 'illicit_recall': 0.6778523489932886, 'illicit_f1': 0.6839729119638827}

Test (EvolveGCN-O):
{'accuracy': 0.9328505007153076, 'illicit_precision': 0.42424242424242425, 'illicit_recall': 0.5062893081761006, 'illicit_f1': 0.46164874551971324}


In [16]:
results_o = pd.DataFrame([
    {
        "Feature Set": "Temporal Graph Snapshots",
        "Model": "EvolveGCN-O",
        "Num Features": in_channels,
        "Split": "validation",
        "Accuracy": val_metrics_o["accuracy"],
        "Illicit Precision": val_metrics_o["illicit_precision"],
        "Illicit Recall": val_metrics_o["illicit_recall"],
        "Illicit F1": val_metrics_o["illicit_f1"],
        "Runtime Minutes": runtime_minutes_o
    },
    {
        "Feature Set": "Temporal Graph Snapshots",
        "Model": "EvolveGCN-O",
        "Num Features": in_channels,
        "Split": "test",
        "Accuracy": test_metrics_o["accuracy"],
        "Illicit Precision": test_metrics_o["illicit_precision"],
        "Illicit Recall": test_metrics_o["illicit_recall"],
        "Illicit F1": test_metrics_o["illicit_f1"],
        "Runtime Minutes": runtime_minutes_o
    }
])

results_o

,Feature Set,Model,Num Features,Split,Accuracy,Illicit Precision,Illicit Recall,Illicit F1,Runtime Minutes
0,Temporal Graph Snapshots,EvolveGCN-O,169,validation,0.948961,0.690205,0.677852,0.683973,0.436015
1,Temporal Graph Snapshots,EvolveGCN-O,169,test,0.932851,0.424242,0.506289,0.461649,0.436015


### Results

EvolveGCN-O achieved an illicit F1 score of `0.684` on the validation set and `0.462` on the test set. These results represent a substantial improvement over EvolveGCN-H, demonstrating stronger generalization to unseen temporal data.

The total training and evaluation runtime was `0.476` minutes, which was substantially faster than the previous machine learning and graph neural network models evaluated in this project. By evolving the graph convolutional weights rather than the hidden node representations, EvolveGCN-O was better able to capture temporal changes in the transaction graph. Although EvolveGCN-O did not surpass the Random Forest baseline, it demonstrated that temporal graph neural networks can learn meaningful temporal representations for fraud detection on the Elliptic dataset.

In [17]:
# combine 01 to 05
results_05 = pd.concat(
    [
        results_h,
        results_o
    ],
    ignore_index=True
)


In [18]:
comparison_columns = [
    "Feature Set",
    "Model",
    "Num Features",
    "Accuracy",
    "Illicit Precision",
    "Illicit Recall",
    "Illicit F1",
    "Runtime Minutes"
]

results_05_validation = (
    results_05
    .loc[results_05["Split"] == "validation", comparison_columns]
    .copy()
)

comparison_05 = pd.concat(
    [
        comparison_04,
        results_05_validation
    ],
    ignore_index=True
)

In [19]:
comparison_05 = (
    comparison_05
    .sort_values("Illicit F1", ascending=False)
    .reset_index(drop=True)
)

display(comparison_05)

,Feature Set,Model,Num Features,Accuracy,Illicit Precision,Illicit Recall,Illicit F1,Runtime Minutes
0,Original Features,Random Forest,169,0.990339,0.994975,0.885906,0.937278,0.694000
1,Node2Vec Embeddings,Random Forest,201,0.989245,0.989899,0.876957,0.930012,0.933000
2,DeepWalk Embeddings,Random Forest,201,0.989245,0.992386,0.874720,0.929845,0.927000
3,Original + Graph Features,GraphSAGE,169,0.971382,0.909605,0.720358,0.803995,8.638000
4,Original Features Only,GraphSAGE,165,0.968647,0.874659,0.718121,0.788698,8.635000
5,Temporal Graph Snapshots,EvolveGCN-O,169,0.948961,0.690205,0.677852,0.683973,0.436015
6,Original + Graph Features,GCN,169,0.952971,0.853933,0.510067,0.638655,6.133000
7,Original Features Only,GCN,165,0.948232,0.802974,0.483221,0.603352,5.820000
8,Temporal Graph Snapshots,EvolveGCN-H,169,0.909953,0.464448,0.686801,0.554152,0.477907
9,Node2Vec Embeddings,Logistic Regression,201,0.784360,0.269424,0.961969,0.420950,0.094000


In [20]:
# save
results_dir = "/content/drive/MyDrive/Elliptic/results/"

comparison_05.to_csv(
    os.path.join(results_dir, "05_model_comparison.csv"),
    index=False
)

## Summary

We evaluated two temporal graph neural network architectures, EvolveGCN-H and EvolveGCN-O, on the Elliptic Bitcoin transaction dataset. Both models successfully learned temporal graph representations from sequential transaction snapshots. Among the two approaches, EvolveGCN-O consistently outperformed EvolveGCN-H, demonstrating that evolving the graph convolution weights was more effective than evolving the hidden representations for this dataset.

Although the temporal GNNs captured meaningful temporal structure, the previously developed Random Forest baseline continued to achieve the strongest overall fraud detection performance. This suggests that the engineered transaction features remain highly informative for the Elliptic dataset. Future work could explore additional hyperparameter tuning, longer training schedules, and more advanced temporal architectures such as Temporal Graph Networks (TGN).



## Next Steps

The next notebook investigates Topological Data Analysis (TDA) as an alternative approach for fraud detection. Unlike graph neural networks, TDA characterizes the underlying shape and connectivity of transaction networks through topological features. This experiment will examine whether topological representations provide complementary information for identifying illicit Bitcoin transactions for `06_Topological_Data_Analysis.ipynb`.